# MCMO - Exploration and Testing

**Objetivo:** Explorar o repositório MCMO e testar em datasets

**Paper:** Reduced-space Multistream Classification based on Multi-objective Evolutionary Optimization (IEEE TEVC 2023)

**GitHub:** https://github.com/Jesen-BT/MCMO

**Data:** 2025-11-23

## 1. Setup Inicial

Clonar repositório e verificar estrutura

In [ ]:
# Clone repositório
!git clone https://github.com/Jesen-BT/MCMO
%cd MCMO

In [ ]:
# Explorar estrutura do repositório
!ls -la

In [ ]:
# Verificar se tem README
!cat README.md 2>/dev/null || echo "No README found"

In [ ]:
# Verificar se tem requirements
!cat requirements.txt 2>/dev/null || echo "No requirements.txt found"

In [ ]:
# Listar arquivos Python
!find . -name "*.py" -type f

## 2. Instalação de Dependências

Instalar dependências do MCMO

In [ ]:
# Instalar dependências comuns do paper
# Baseado no paper, MCMO provavelmente precisa:
# - scikit-learn (GMM, metrics)
# - DEAP (NSGA-II)
# - numpy, pandas
# - scikit-multiflow (Hoeffding Tree)

!pip install scikit-learn
!pip install deap
!pip install numpy pandas
!pip install scikit-multiflow
!pip install scipy

In [ ]:
# Se houver requirements.txt, instalar
!pip install -r requirements.txt 2>/dev/null || echo "No requirements.txt, usando dependências padrão"

## 3. Exploração do Código

Entender a estrutura e API do MCMO

In [ ]:
# Tentar importar MCMO
import sys
sys.path.insert(0, '.')

try:
    import MCMO
    print("✓ MCMO module found!")
    print(f"\nMCMO attributes:")
    print([attr for attr in dir(MCMO) if not attr.startswith('_')])
except ImportError as e:
    print(f"✗ MCMO module not found: {e}")
    print("\nTrying alternative imports...")
    
    # Tentar importar de subdiretórios
    try:
        from src import MCMO
        print("✓ Found in src/")
    except:
        try:
            from mcmo import MCMO
            print("✓ Found as mcmo package")
        except:
            print("✗ Could not find MCMO module")

In [ ]:
# Listar arquivos Python e ler o principal
import os

# Encontrar arquivo principal
python_files = []
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.py'):
            python_files.append(os.path.join(root, file))

print("Python files found:")
for f in python_files:
    print(f"  {f}")

In [ ]:
# Se houver main.py ou MCMO.py, ler as primeiras linhas
main_candidates = ['MCMO.py', 'main.py', 'src/MCMO.py', 'mcmo/mcmo.py']

for candidate in main_candidates:
    if os.path.exists(candidate):
        print(f"\n{'='*60}")
        print(f"Reading {candidate}:")
        print(f"{'='*60}")
        !head -100 {candidate}
        break

## 4. Análise da API

Se o código estiver disponível, entender como usar

In [ ]:
# Buscar por classes principais
!grep -n "class" *.py 2>/dev/null || echo "No Python files in root"
!grep -rn "class" . --include="*.py" | head -20

In [ ]:
# Buscar por funções fit, predict, train
!grep -rn "def fit" . --include="*.py" | head -10
!grep -rn "def predict" . --include="*.py" | head -10
!grep -rn "def train" . --include="*.py" | head -10

## 5. Implementação Fallback

Se o repositório não tiver código completo, implementar componentes básicos do zero

In [ ]:
# Implementação simplificada do MCMO baseada no paper
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from deap import base, creator, tools, algorithms

class MCMOSimplified:
    """
    Implementação simplificada do MCMO baseada no paper.
    
    Componentes:
    1. Feature Selection via NSGA-II (simplificado)
    2. GMM-based Sample Weighting
    3. Ensemble de Base Classifiers
    """
    
    def __init__(self, n_sources=3, k_gaussians=7, n_features_select=None):
        self.n_sources = n_sources
        self.k_gaussians = k_gaussians
        self.n_features_select = n_features_select
        
        self.selected_features_ = None
        self.gmm_ = None
        self.base_classifiers_ = []
        self.classifier_weights_ = []
    
    def _feature_selection_simple(self, sources, target):
        """
        Feature selection simplificado (sem NSGA-II completo).
        Usa correlation-based filter.
        """
        n_features = sources[0][0].shape[1]
        
        if self.n_features_select is None:
            # Select top 70% features by default
            self.n_features_select = int(n_features * 0.7)
        
        # Concatenar todas source streams
        X_all = np.vstack([s[0] for s in sources])
        y_all = np.hstack([s[1] for s in sources])
        
        # Calcular correlação com labels (discriminative power)
        from scipy.stats import pearsonr
        correlations = []
        for f in range(n_features):
            try:
                corr, _ = pearsonr(X_all[:, f], y_all)
                correlations.append(abs(corr))
            except:
                correlations.append(0.0)
        
        # Selecionar top features
        selected_idx = np.argsort(correlations)[-self.n_features_select:]
        self.selected_features_ = sorted(selected_idx)
        
        print(f"Feature selection: {len(self.selected_features_)}/{n_features} features selected")
        return self.selected_features_
    
    def _build_gmm(self, target_X):
        """
        Construir GMM no target stream.
        """
        # Aplicar feature selection
        target_X_selected = target_X[:, self.selected_features_]
        
        # Fit GMM
        self.gmm_ = GaussianMixture(
            n_components=self.k_gaussians,
            covariance_type='full',
            random_state=42
        )
        self.gmm_.fit(target_X_selected)
        
        print(f"GMM fitted with {self.k_gaussians} components")
    
    def _compute_sample_weights(self, source_X):
        """
        Computar weights para source samples usando GMM.
        """
        # Aplicar feature selection
        source_X_selected = source_X[:, self.selected_features_]
        
        # Calcular conditional probability para cada sample
        # score_samples retorna log-likelihood, convertemos para prob
        log_probs = self.gmm_.score_samples(source_X_selected)
        probs = np.exp(log_probs)
        
        # Normalizar weights
        weights = probs / probs.sum()
        
        return weights
    
    def fit_predict(self, sources, target_X):
        """
        Fit MCMO e predizer labels do target.
        
        Args:
            sources: List of (X, y) tuples for each source stream
            target_X: Target stream features (no labels)
        
        Returns:
            predictions: Array of predicted labels
        """
        print(f"\nMCMO Fitting:")
        print(f"  Sources: {len(sources)} streams")
        print(f"  Target: {target_X.shape[0]} samples")
        
        # 1. Feature Selection
        print("\n1. Feature Selection...")
        self._feature_selection_simple(sources, target_X)
        
        # 2. Build GMM on target
        print("\n2. Building GMM on target...")
        self._build_gmm(target_X)
        
        # 3. Train base classifiers on weighted sources
        print("\n3. Training base classifiers...")
        self.base_classifiers_ = []
        self.classifier_weights_ = []
        
        for i, (source_X, source_y) in enumerate(sources):
            # Compute weights
            weights = self._compute_sample_weights(source_X)
            
            # Train classifier
            clf = DecisionTreeClassifier(max_depth=10, random_state=42)
            
            # Aplicar feature selection
            source_X_selected = source_X[:, self.selected_features_]
            
            clf.fit(source_X_selected, source_y, sample_weight=weights)
            
            self.base_classifiers_.append(clf)
            self.classifier_weights_.append(weights.mean())
            
            print(f"  Source {i+1}: {source_X.shape[0]} samples, ")
                  f"avg_weight={weights.mean():.4f}")
        
        # Normalizar classifier weights
        total_weight = sum(self.classifier_weights_)
        self.classifier_weights_ = [w/total_weight for w in self.classifier_weights_]
        
        # 4. Ensemble prediction
        print("\n4. Ensemble prediction...")
        target_X_selected = target_X[:, self.selected_features_]
        
        # Predições de cada base classifier
        predictions_list = []
        for clf in self.base_classifiers_:
            pred = clf.predict(target_X_selected)
            predictions_list.append(pred)
        
        # Weighted voting
        predictions_array = np.array(predictions_list)  # shape: (n_classifiers, n_samples)
        
        # Para cada sample, votar com pesos
        final_predictions = []
        for i in range(target_X.shape[0]):
            votes = predictions_array[:, i]
            
            # Contar votos ponderados
            unique_classes = np.unique(votes)
            weighted_votes = {}
            
            for cls in unique_classes:
                weighted_vote = sum([
                    self.classifier_weights_[j] 
                    for j in range(len(votes)) 
                    if votes[j] == cls
                ])
                weighted_votes[cls] = weighted_vote
            
            # Classe com maior voto ponderado
            final_pred = max(weighted_votes, key=weighted_votes.get)
            final_predictions.append(final_pred)
        
        print(f"\n✓ MCMO prediction completed")
        return np.array(final_predictions)

print("✓ MCMOSimplified class defined")

## 6. Teste com Dados Sintéticos

Testar MCMO com dados simples para verificar funcionamento

In [ ]:
# Gerar dados sintéticos multistream
from sklearn.datasets import make_classification

def generate_multistream_data(n_sources=3, n_samples=500, n_features=20, n_classes=2):
    """
    Gera dados multistream sintéticos com covariate shift.
    """
    sources = []
    
    for i in range(n_sources):
        # Cada source tem distribuição ligeiramente diferente
        X, y = make_classification(
            n_samples=n_samples,
            n_features=n_features,
            n_informative=int(n_features * 0.7),
            n_redundant=int(n_features * 0.2),
            n_classes=n_classes,
            random_state=42 + i,
            flip_y=0.05 * i  # Adicionar ruído crescente
        )
        
        # Adicionar shift na distribuição
        X = X + np.random.randn(*X.shape) * 0.1 * i
        
        sources.append((X, y))
    
    # Target stream com mais shift
    X_target, y_target = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=int(n_features * 0.7),
        n_redundant=int(n_features * 0.2),
        n_classes=n_classes,
        random_state=42 + n_sources,
        flip_y=0.1
    )
    X_target = X_target + np.random.randn(*X_target.shape) * 0.3
    
    return sources, X_target, y_target

# Gerar dados
print("Generating synthetic multistream data...")
sources, X_target, y_target = generate_multistream_data(
    n_sources=3,
    n_samples=500,
    n_features=20,
    n_classes=2
)

print(f"\nGenerated:")
for i, (X, y) in enumerate(sources):
    print(f"  Source {i+1}: {X.shape}, classes: {np.unique(y)}")
print(f"  Target: {X_target.shape}, classes: {np.unique(y_target)}")

In [ ]:
# Testar MCMO simplificado
print("Testing MCMO...\n")

mcmo = MCMOSimplified(n_sources=3, k_gaussians=5, n_features_select=14)
predictions = mcmo.fit_predict(sources, X_target)

# Avaliar
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_target, predictions)
f1 = f1_score(y_target, predictions, average='weighted')

print(f"\n{'='*60}")
print("MCMO Results on Synthetic Data:")
print(f"{'='*60}")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_target, predictions))

## 7. Teste com Electricity Dataset

Testar em dataset real usado no paper

In [ ]:
# Baixar Electricity dataset
from river.datasets import Elec2

print("Loading Electricity dataset...")
dataset = Elec2()

# Coletar samples
X_list = []
y_list = []

for i, (x, y) in enumerate(dataset):
    if i >= 6000:  # Limitar a 6000 samples
        break
    X_list.append(list(x.values()))
    y_list.append(y)

X_elec = np.array(X_list)
y_elec = np.array(y_list)

print(f"Electricity dataset: {X_elec.shape}")
print(f"Classes: {np.unique(y_elec)}")
print(f"Class distribution: {np.bincount(y_elec)}")

In [ ]:
# Criar multistream do Electricity (temporal splitting)
def create_temporal_multistream(X, y, n_sources=3, chunk_size=1000):
    """
    Cria multistream via temporal splitting.
    
    Sources são chunks consecutivos no passado,
    Target é chunk mais recente.
    """
    sources = []
    
    for i in range(n_sources):
        start_idx = i * chunk_size
        end_idx = (i + n_sources) * chunk_size
        
        X_source = X[start_idx:end_idx]
        y_source = y[start_idx:end_idx]
        
        sources.append((X_source, y_source))
    
    # Target = último chunk
    target_start = n_sources * chunk_size
    target_end = target_start + chunk_size
    
    X_target = X[target_start:target_end]
    y_target = y[target_start:target_end]
    
    return sources, X_target, y_target

# Criar multistream
sources_elec, X_target_elec, y_target_elec = create_temporal_multistream(
    X_elec, y_elec,
    n_sources=3,
    chunk_size=1000
)

print("Electricity multistream created:")
for i, (X, y) in enumerate(sources_elec):
    print(f"  Source {i+1}: {X.shape}")
print(f"  Target: {X_target_elec.shape}")

In [ ]:
# Testar MCMO em Electricity
print("Testing MCMO on Electricity...\n")

mcmo_elec = MCMOSimplified(
    n_sources=3,
    k_gaussians=7,
    n_features_select=6  # Electricity tem 8 features, selecionar 6
)

predictions_elec = mcmo_elec.fit_predict(sources_elec, X_target_elec)

# Avaliar
accuracy_elec = accuracy_score(y_target_elec, predictions_elec)
f1_elec = f1_score(y_target_elec, predictions_elec, average='weighted')

# Calcular G-mean
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_target_elec, predictions_elec)
recalls = cm.diagonal() / cm.sum(axis=1)
gmean_elec = np.prod(recalls) ** (1/len(recalls)) if all(recalls > 0) else 0.0

print(f"\n{'='*60}")
print("MCMO Results on Electricity:")
print(f"{'='*60}")
print(f"G-mean:   {gmean_elec:.4f}")
print(f"Accuracy: {accuracy_elec:.4f}")
print(f"F1-score: {f1_elec:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_target_elec, predictions_elec))
print(f"\nConfusion Matrix:")
print(cm)

## 8. Comparação com Baseline Simples

Comparar MCMO com classificador simples (sem multistream)

In [ ]:
# Baseline: Decision Tree treinado em todos source data concatenados
from sklearn.tree import DecisionTreeClassifier

print("Training baseline (single classifier on all sources)...")

# Concatenar todas sources
X_all_sources = np.vstack([s[0] for s in sources_elec])
y_all_sources = np.hstack([s[1] for s in sources_elec])

# Treinar
baseline_clf = DecisionTreeClassifier(max_depth=10, random_state=42)
baseline_clf.fit(X_all_sources, y_all_sources)

# Predizer
predictions_baseline = baseline_clf.predict(X_target_elec)

# Avaliar
accuracy_baseline = accuracy_score(y_target_elec, predictions_baseline)
f1_baseline = f1_score(y_target_elec, predictions_baseline, average='weighted')

cm_baseline = confusion_matrix(y_target_elec, predictions_baseline)
recalls_baseline = cm_baseline.diagonal() / cm_baseline.sum(axis=1)
gmean_baseline = np.prod(recalls_baseline) ** (1/len(recalls_baseline)) if all(recalls_baseline > 0) else 0.0

print(f"\nBaseline Results:")
print(f"  G-mean:   {gmean_baseline:.4f}")
print(f"  Accuracy: {accuracy_baseline:.4f}")
print(f"  F1-score: {f1_baseline:.4f}")

In [ ]:
# Comparação
print(f"\n{'='*60}")
print("COMPARISON: MCMO vs Baseline")
print(f"{'='*60}")
print(f"\n{'Metric':<15} {'MCMO':<12} {'Baseline':<12} {'Diff':<12}")
print("-" * 60)
print(f"{'G-mean':<15} {gmean_elec:<12.4f} {gmean_baseline:<12.4f} {gmean_elec-gmean_baseline:+.4f}")
print(f"{'Accuracy':<15} {accuracy_elec:<12.4f} {accuracy_baseline:<12.4f} {accuracy_elec-accuracy_baseline:+.4f}")
print(f"{'F1-score':<15} {f1_elec:<12.4f} {f1_baseline:<12.4f} {f1_elec-f1_baseline:+.4f}")

if gmean_elec > gmean_baseline:
    print(f"\n✓ MCMO outperforms baseline by {(gmean_elec-gmean_baseline)*100:.2f}% in G-mean")
elif gmean_elec < gmean_baseline:
    print(f"\n✗ Baseline outperforms MCMO by {(gmean_baseline-gmean_elec)*100:.2f}% in G-mean")
else:
    print(f"\n= MCMO and baseline have similar performance")

## 9. Análise de Componentes

Analisar componentes do MCMO (feature selection, GMM)

In [ ]:
# Analisar features selecionados
print("MCMO Components Analysis:")
print(f"\n1. Feature Selection:")
print(f"   Selected {len(mcmo_elec.selected_features_)}/{X_elec.shape[1]} features")
print(f"   Features: {mcmo_elec.selected_features_}")

# Analisar GMM
print(f"\n2. GMM:")
print(f"   K={mcmo_elec.k_gaussians} components")
print(f"   Converged: {mcmo_elec.gmm_.converged_}")
print(f"   Iterations: {mcmo_elec.gmm_.n_iter_}")

# Analisar classifier weights
print(f"\n3. Ensemble Weights:")
for i, weight in enumerate(mcmo_elec.classifier_weights_):
    print(f"   Source {i+1}: {weight:.4f}")

## 10. Exportar Resultados

Salvar resultados para análise posterior

In [ ]:
# Criar summary de resultados
import pandas as pd

results_summary = pd.DataFrame([
    {
        'Model': 'MCMO',
        'Dataset': 'Electricity',
        'G-mean': gmean_elec,
        'Accuracy': accuracy_elec,
        'F1-score': f1_elec,
        'N_features_selected': len(mcmo_elec.selected_features_),
        'N_features_total': X_elec.shape[1]
    },
    {
        'Model': 'Baseline',
        'Dataset': 'Electricity',
        'G-mean': gmean_baseline,
        'Accuracy': accuracy_baseline,
        'F1-score': f1_baseline,
        'N_features_selected': X_elec.shape[1],
        'N_features_total': X_elec.shape[1]
    }
])

print("\nResults Summary:")
print(results_summary.to_string(index=False))

# Salvar CSV
results_summary.to_csv('mcmo_electricity_results.csv', index=False)
print("\n✓ Results saved to mcmo_electricity_results.csv")

## 11. Conclusões

Sumário da exploração

In [ ]:
print("="*60)
print("MCMO EXPLORATION SUMMARY")
print("="*60)

print("\n✓ COMPLETED:")
print("  - Cloned MCMO repository")
print("  - Implemented MCMOSimplified (fallback)")
print("  - Tested on synthetic data")
print("  - Tested on Electricity dataset")
print("  - Compared with baseline")

print("\n📊 KEY FINDINGS:")
print(f"  - MCMO G-mean: {gmean_elec:.4f}")
print(f"  - Baseline G-mean: {gmean_baseline:.4f}")
print(f"  - Difference: {(gmean_elec-gmean_baseline)*100:+.2f}%")

print("\n🔧 COMPONENTS WORKING:")
print("  ✓ Feature Selection (correlation-based)")
print("  ✓ GMM-based Sample Weighting")
print("  ✓ Ensemble Classification")
print("  ✗ NSGA-II (usando simplificação)")
print("  ✗ Drift Detection (não implementado ainda)")

print("\n📝 NEXT STEPS:")
print("  1. Verificar se repositório tem NSGA-II completo")
print("  2. Implementar drift detection")
print("  3. Testar em mais datasets")
print("  4. Integrar ao pipeline DSL-AG-hybrid")
print("  5. Comparar com GBML/ARF/HAT/SRP")

print("\n" + "="*60)